# GPAT

Gridded Plume Analysis Tool (GPAT) modelling framework. This simulates flight trajectories, estimates fuel burn and emissions, models dispersion effects, and aggregates plume data to a common Eulerian grid for further photochemical and microphysical processing.

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from dataclasses import asdict
from pycontrails.models.gpat.gpat import GPAT, SimParams, FlParams, PlumeParams, MetParams, ChemParams, dict_to_dataclass
from pycontrails.models.gpat.plume_vis import animate_plume
import os
import holoviews as hv
import hvplot.pandas
import hvplot.xarray

In [2]:
# global simulation parameters
sim_params = {
    "t_fl": (pd.to_datetime("2022-01-20 13:00:00"), pd.Timedelta(minutes=1), pd.Timedelta(hours=1)),# (start time, time step, run time)
    "t_pl": (pd.to_datetime("2022-01-20 13:00:00"), pd.Timedelta(minutes=1), pd.Timedelta(hours=2)),# (start time, time step, max age)
    "t_sim": (pd.to_datetime("2022-01-20 12:00:00"), pd.Timedelta(seconds=20), pd.Timedelta(hours=4)),# (start time, time step, run time)
    "t_out": (pd.to_datetime("2022-01-20 12:00:00"), pd.Timedelta(minutes=5), pd.Timedelta(hours=4)),# (start time, time step, run time)
    "lat_bounds": (0.0, 1.0),  # lat bounds [deg]
    "lon_bounds": (0.0, 1.0),  # lon bounds [deg]
    "alt_bounds": (10000, 11000),  # alt bounds [m]
    "hres_sim_c": 0.05,  # coarse horizontal resolution [deg]
    "vres_sim_c": 500,  # coarse vertical resolution [m]
    "hres_sim_f": 0.001,  # fine horizontal resolution [deg]
    "vres_sim_f": 100,  # fine vertical resolution [m]

    "run_path": "/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/",
    "data_path": "/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/", # "/projects/Impact_of_aviation_on_climate
    "job_id": "GPAT_Jan_2026_test_2_ac",
}

In [3]:
#flight trajectory parameters
fl_params = {
    "mode": "synthetic",
    "file": None,  # flight trajectory file

    "ac_type": "A320",  # aircraft type
    "fl0_speed": 100.0,  # m/s
    "fl0_heading": 45.0,  # deg
    "fl0_coords0": (0.1, 0.1, 10500),  # lat, lon, alt [deg, deg, m]
    "sep_dist": (10000, 5000, 0),  # dx, dy, dz [m]
    "n_ac": 2,  # number of aircraft
}

In [4]:
# plume dispersion parameters
plume_params = {
    "depth": 50.0,  # initial plume depth, [m]
    "width": 50.0,  # initial plume width, [m]
    "verbose_outputs": False,  # print verbose outputs
    "n_slices": 5,  # number of slices in the plume
    "shear": 0.01,  # shear [m/s]
    }

In [5]:
# meteorology parameters
met_params = {
    "eastward_wind": 5.0,  # m/s
    "northward_wind": 3.0,  # m/s
    "lagrangian_tendency_of_air_pressure": 0.0,  # m/s
}

In [6]:
# chemistry parameters
chem_params = {
    "run_chem": True,
    "species_emi": ("NO",),
    "species_plume": ("NO", "NO2", "O3", "NO3", "N2O5",
                      "HNO3", "HONO", "HO2NO2","PAN", 
                      "CH3O2NO2","H2O2", "CH3OOH",
                      "CO", "CH4", "HCHO", "SO2", "SA"),
    "species_out": ("O3", "NO2", "NO", "NO3", "N2O5", 
                    "HNO3", "HONO", "HO2", "OH", "H2O2",
                    "CO", "CH4", "CH3O2","HO2NO2", "PAN", "SO2" )
}

In [7]:
sim_params = SimParams(**sim_params)
fl_params = FlParams(**fl_params)
plume_params = PlumeParams(**plume_params)
met_params = MetParams(**met_params)
chem_params = ChemParams(**chem_params)

gpat = GPAT(sim_params, fl_params, plume_params, met_params, chem_params)


In [8]:
gpat.preprocess_gpat()

/home/ktait98/miniconda3/envs/contrails/lib/python3.12/site-packages/xarray/core/duck_array_ops.py:234: UserWarning: no explicit representation of timezones available for np.datetime64
  return data.astype(dtype, **kwargs)


flight 0 done
flight 1 done
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/inputs/GPAT_Jan_2026_test_2_ac/boxm_ds.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/inputs/GPAT_Jan_2026_test_2_ac/fl_ds.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/inputs/GPAT_Jan_2026_test_2_ac/pl_ds.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/outputs/GPAT_Jan_2026_test_2_ac/boxm_out.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/outputs/GPAT_Jan_2026_test_2_ac/patch_table.nc
Saved /home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/outputs/GPAT_Jan_2026_test_2_ac/pl_out.nc


/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/gpat.py:836: RuntimeWarning: invalid value encountered in cast
  age_seconds = (age_values / np.timedelta64(1, 's')).astype(int)
/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/gpat.py:839: RuntimeWarning: invalid value encountered in cast
  age_seconds = np.where(np.isnat(age_values), 0, (age_values / np.timedelta64(1, 's')).astype(int))


In [9]:
fl_ds = xr.open_dataset(f"{gpat.inputs_job}/fl_ds.nc")
fl_ds

<xarray.Dataset> Size: 10kB
Dimensions:               (flight_id: 2, waypoint: 24)
Coordinates:
  * flight_id             (flight_id) int64 16B 0 1
  * waypoint              (waypoint) int64 192B 0 1 2 3 4 5 ... 19 20 21 22 23
Data variables: (12/16)
    longitude             (flight_id, waypoint) float64 384B ...
    latitude              (flight_id, waypoint) float64 384B ...
    level                 (flight_id, waypoint) float64 384B ...
    altitude              (flight_id, waypoint) float64 384B ...
    time                  (flight_id, waypoint) <U20 4kB ...
    true_airspeed         (flight_id, waypoint) float64 384B ...
    ...                    ...
    thrust                (flight_id, waypoint) float64 384B ...
    rocd                  (flight_id, waypoint) float64 384B ...
    fuel_flow_per_engine  (flight_id, waypoint) float64 384B ...
    thrust_setting        (flight_id, waypoint) float64 384B ...
    time_rel_s            (flight_id, waypoint) int64 384B ...
    time_idx              (flight_id, waypoint) int64 384B ...
Attributes:
    description:  Flight trajectory and emissions data for BOXM

In [10]:
pl_ds = xr.open_dataset(f"{gpat.inputs_job}/pl_ds.nc")
pl_ds

<xarray.Dataset> Size: 1MB
Dimensions:           (flight_id: 2, waypoint: 23, time: 134, species_emi: 1)
Coordinates:
  * flight_id         (flight_id) int64 16B 0 1
  * waypoint          (waypoint) int64 184B 0 1 2 3 4 5 6 ... 17 18 19 20 21 22
  * time              (time) <U20 11kB '2022-01-20T13:00:00Z' ... '2022-01-20...
  * species_emi       (species_emi) <U2 8B 'NO'
    time_rel_s        (time) int64 1kB ...
    time_idx          (time) int64 1kB ...
Data variables: (12/13)
    age               (flight_id, waypoint, time) <U21 518kB ...
    longitude         (flight_id, waypoint, time) float64 49kB ...
    latitude          (flight_id, waypoint, time) float64 49kB ...
    level             (flight_id, waypoint, time) float64 49kB ...
    width             (flight_id, waypoint, time) float64 49kB ...
    depth             (flight_id, waypoint, time) float64 49kB ...
    ...                ...
    sigma_yy          (flight_id, waypoint, time) float64 49kB ...
    sigma_yz          (flight_id, waypoint, time) float64 49kB ...
    sigma_zz          (flight_id, waypoint, time) float64 49kB ...
    altitude          (flight_id, waypoint, time) float64 49kB ...
    species_emi_mass  (flight_id, waypoint, species_emi) float64 368B ...
    age_s             (flight_id, waypoint, time) int64 49kB ...
Attributes:
    ts_fl:              60.0
    ts_pl:              60.0
    ts_sim:             20.0
    species_emi:        NO
    species_plume:      ['NO', 'NO2', 'O3', 'NO3', 'N2O5', 'HNO3', 'HONO', 'H...
    species_emi_num:    8
    species_plume_num:  [  8   4   6   5   7  14  13  15 198 217  12 144  11 ...
    description:        Emission species mass in plume segments

In [11]:
boxm_ds = xr.open_dataset(f"{gpat.inputs_job}/boxm_ds.nc")

In [12]:
gpat.boxm_ds_stacked

<xarray.Dataset> Size: 29MB
Dimensions:          (time: 721, cell: 800, species_boxm: 219)
Coordinates:
  * time             (time) object 6kB '2022-01-20T12:00:00Z' ... '2022-01-20...
    air_pressure     (cell) float64 6kB 2.354e+04 2.354e+04 ... 2.544e+04
    altitude         (cell) float64 6kB 1.075e+04 1.075e+04 ... 1.025e+04
  * species_boxm     (species_boxm) <U10 9kB 'O1D' 'O' 'OH' ... 'EMPOA' 'P2007'
    time_rel_s       (time) int64 6kB 0 20 40 60 80 ... 14340 14360 14380 14400
    time_idx         (time) int64 6kB 1 2 3 4 5 6 7 ... 716 717 718 719 720 721
    level            (cell) float64 6kB 235.4 235.4 235.4 ... 254.4 254.4 254.4
    longitude        (cell) float64 6kB 0.025 0.025 0.025 ... 0.975 0.975 0.975
    latitude         (cell) float64 6kB 0.025 0.075 0.125 ... 0.875 0.925 0.975
Dimensions without coordinates: cell
Data variables:
    air_temperature  (cell, time) float64 5MB 226.8 226.8 226.8 ... 231.5 231.5
    H2O              (cell, time) float64 5MB 2.936e+15 2.936e+15 ... 3.788e+15
    M                (cell, time) float64 5MB 7.518e+18 7.518e+18 ... 7.959e+18
    O2               (cell, time) float64 5MB 1.563e+18 1.563e+18 ... 1.655e+18
    N2               (cell, time) float64 5MB 5.87e+18 5.87e+18 ... 6.215e+18
    sza              (cell, time) float64 5MB 0.3539 0.3537 ... 1.06 1.062
    bg_chem          (cell, species_boxm) float64 1MB 0.0 0.0 0.0 ... 0.0 0.0
Attributes: (12/19)
    ts_fl:              60.0
    ts_pl:              60.0
    ts_sim:             20.0
    hres_sim_c:         0.05
    vres_sim_c:         500
    hres_sim_f:         0.001
    ...                 ...
    photol_params:      57
    photol_coeffs:      96
    therm_coeffs:       512
    flux_species:       130
    description:        BOXM coarse-grid meteorology and background chemistry...
    note:               Emissions and plume segments handled separately via P...

In [13]:
pl_out = xr.open_dataset(f"{gpat.outputs_job}/pl_out.nc")

pl_out

<xarray.Dataset> Size: 1MB
Dimensions:             (flight_id: 2, waypoint: 23, time: 134,
                         species_plume: 17)
Coordinates:
  * flight_id           (flight_id) int64 16B 0 1
  * waypoint            (waypoint) int64 184B 0 1 2 3 4 5 ... 17 18 19 20 21 22
  * time                (time) <U20 11kB '2022-01-20T13:00:00Z' ... '2022-01-...
    time_rel_s          (time) int64 1kB ...
    time_idx            (time) int64 1kB ...
  * species_plume       (species_plume) <U8 544B 'NO' 'NO2' 'O3' ... 'SO2' 'SA'
Data variables:
    latitude            (flight_id, waypoint, time) float64 49kB ...
    level               (flight_id, waypoint, time) float64 49kB ...
    width               (flight_id, waypoint, time) float64 49kB ...
    depth               (flight_id, waypoint, time) float64 49kB ...
    heading             (flight_id, waypoint, time) float64 49kB ...
    sigma_yy            (flight_id, waypoint, time) float64 49kB ...
    sigma_yz            (flight_id, waypoint, time) float64 49kB ...
    sigma_zz            (flight_id, waypoint, time) float64 49kB ...
    altitude            (flight_id, waypoint, time) float64 49kB ...
    species_plume_mass  (flight_id, waypoint, species_plume, time) float64 838kB ...
Attributes:
    ts_fl:              60.0
    ts_pl:              60.0
    ts_sim:             20.0
    species_emi:        NO
    species_plume:      ['NO', 'NO2', 'O3', 'NO3', 'N2O5', 'HNO3', 'HONO', 'H...
    species_emi_num:    8
    species_plume_num:  [  8   4   6   5   7  14  13  15 198 217  12 144  11 ...
    description:        Plume segment output for BOXM

In [14]:
gpat.boxm_out_stacked

<xarray.Dataset> Size: 10MB
Dimensions:      (time: 49, species_out: 16, cell: 800)
Coordinates:
  * time         (time) object 392B '2022-01-20T12:00:00Z' ... '2022-01-20T16...
    time_rel_s   (time) int64 392B 0 300 600 900 ... 13500 13800 14100 14400
    time_idx     (time) int64 392B 1 16 31 46 61 76 ... 646 661 676 691 706 721
    altitude_c   (cell) float64 6kB 1.025e+04 1.025e+04 ... 1.075e+04 1.075e+04
  * species_out  (species_out) <U10 640B 'O3' 'NO2' 'NO' ... 'PAN' 'SO2'
    level_c      (cell) float64 6kB 254.4 254.4 254.4 ... 235.4 235.4 235.4
    longitude_c  (cell) float64 6kB 0.025 0.025 0.025 ... 0.975 0.975 0.975
    latitude_c   (cell) float64 6kB 0.025 0.075 0.125 ... 0.875 0.925 0.975
Dimensions without coordinates: cell
Data variables:
    Y_bg_c       (cell, species_out, time) float64 5MB dask.array<chunksize=(800, 16, 49), meta=np.ndarray>
    Y_del_c      (cell, species_out, time) float64 5MB dask.array<chunksize=(800, 16, 49), meta=np.ndarray>
    active_flag  (cell, time) bool 39kB dask.array<chunksize=(800, 49), meta=np.ndarray>
Attributes:
    description:  BOXM output coarse-grid chemistry fields

In [15]:
gpat.patch_table

<xarray.Dataset> Size: 384B
Dimensions:      (row: 0, species_out: 16)
Coordinates:
  * row          (row) int64 0B 
    patch_id     (row) int64 0B 
  * species_out  (species_out) <U6 384B 'O3' 'NO2' 'NO' ... 'HO2NO2' 'PAN' 'SO2'
    time         (row) object 0B 
    time_rel_s   (row) int64 0B 
    time_idx     (row) int64 0B 
    latitude_f   (row) float64 0B 
    longitude_f  (row) float64 0B 
    altitude_f   (row) float64 0B 
    level_f      (row) float64 0B 
Data variables:
    Y_del_f      (row, species_out) float64 0B dask.array<chunksize=(0, 16), meta=np.ndarray>
Attributes:
    description:  Fine grid plume patch output for BOXM

In [16]:
gpat.bg_chem

<xarray.Dataset> Size: 1MB
Dimensions:       (species_boxm: 219, latitude: 20, longitude: 20, level: 2)
Coordinates:
    month         int64 8B 0
  * species_boxm  (species_boxm) <U10 9kB 'O1D' 'O' 'OH' ... 'EMPOA' 'P2007'
  * longitude     (longitude) float64 160B 0.025 0.075 0.125 ... 0.925 0.975
  * latitude      (latitude) float64 160B 0.025 0.075 0.125 ... 0.925 0.975
  * level         (level) float64 16B 254.4 235.4
Data variables:
    bg_chem       (latitude, longitude, level, species_boxm) float64 1MB 0.0 ...